In [29]:
import pandas as pd


In [30]:
# Lectura de documentos CSV

clientes = pd.read_csv("../data/raw/clientes.csv")
prestamos = pd.read_csv("../data/raw/prestamos.csv")
pagos = pd.read_csv("../data/raw/pagos.csv")

In [31]:
documentos = [clientes, prestamos, pagos]

In [32]:
# Visualización general de documentos en busca de anomalias

print(clientes.describe())
print(prestamos.describe())
print(pagos.describe())

              edad  ingresos_anuales  score_crediticio
count  5000.000000       4841.000000       4907.000000
mean     41.975200      29352.054741        599.664357
std      13.184375      32599.443596         67.721008
min      -1.000000         -1.000000          0.000000
25%      32.000000      19200.000000        557.000000
50%      41.000000      26400.000000        599.000000
75%      50.000000      34800.000000        644.000000
max     135.000000     900000.000000       1000.000000
             importe  plazo_meses  tipo_interes
count    8080.000000  8080.000000   8080.000000
mean    12514.659653    20.621287      9.596079
std     16610.251634    17.645072      2.749527
min       500.000000     6.000000      0.035400
25%      5050.000000     6.000000      7.780000
50%      8850.000000    12.000000      9.530000
75%     15050.000000    24.000000     11.380000
max    400000.000000    84.000000     20.990000
       importe_cuota  importe_pagado  dias_retraso
count   78924.000000  

In [33]:
# Definición de función de limpieza

def limpiar_rango(df, columna, min_val, max_val):
    filas_antes = len(df)
    df_limpio = df[(df[columna] >= min_val) & (df[columna] <= max_val)]
    print(f"[{columna}] Se han eliminado {filas_antes - len(df_limpio)} filas")
    return df_limpio

Se observa en la tabla Clientes valores inconsistentes y diferencias en el numero de Filas.

Decisión: Se hace uso de la fórmula "limpiar_rango" para limitar los datos de Edad e Ingresos anuales, agregando en este último un máximo del triple de la media general.

Se eliminar valores nulos de la columna Score Crediticio igualando así el numero de filas por columna en el documento.

In [34]:
# Aplicación de la función de limpieza

clientes = limpiar_rango(clientes, "edad", 18, 100)
clientes = limpiar_rango(clientes, "ingresos_anuales", 0, clientes["ingresos_anuales"].mean()*3 )
clientes_fuera = ~clientes["score_crediticio"].between(300, 850)
clientes = (limpiar_rango(clientes, "score_crediticio", 300, 850))

[edad] Se han eliminado 6 filas
[ingresos_anuales] Se han eliminado 180 filas
[score_crediticio] Se han eliminado 92 filas


In [35]:
# Se analizan las primeras 15 filas para visualizar formato
print(clientes.head(15))

   id_cliente  fecha_alta  edad               provincia     estado_civil  \
0      C00001  2022-01-02    28               Barcelona           casado   
1      C00002  2022-01-03    45               Barcelona          SOLTERO   
2      C00003  2022-01-04    60                 Sevilla  pareja de hecho   
3      C00004  2022-01-04    26                     Bcn           casado   
4      C00005  2022-01-05    34                A Coruña           casado   
5      C00006  2022-01-09    32  Santa Cruz de Tenerife           casado   
6      C00007  09/01/2022    32                  Madrid           casado   
7      C00008  2022-01-10    48                 Sevilla            viudo   
8      C00009  2022-01-12    59                 Sevilla          soltero   
9      C00010  2022-01-14    59                  Madrid           casado   
10     C00011  2022-01-15    24              Pontevedra           casado   
11     C00012  2022-01-17    40                  Madrid          soltero   
12     C0001

Se encuentran anomalias de formato en diferentes columnas del documento Clientes

Decisión: Se crea fórmula para modificar el formato de Fecha, reemplazamos siglas por nombre completo y convertimos texto a mayusculas. 

In [36]:
def limpieza_clientes(df):
    # 1. Conversion de datos a tipo Fecha formato EEUU
    df["fecha_alta"] = pd.to_datetime(df["fecha_alta"], format="mixed", dayfirst=False)

    # 2. Reemplazo de siglas por nombre completo
    df["provincia"] = df["provincia"].replace({
        'bcn': 'barcelona',
        'BCN' : 'barcelona'
        })

    # 3. Mayusculas para texto y elimicación espacios
    df["estado_civil"] = df["estado_civil"].str.upper()
    df["situacion_laboral"] = df["situacion_laboral"].str.upper().str.strip()
    df["provincia"] = df["provincia"].str.upper().str.strip()

    print("Los datos han sido modificados")
    print(df.head(15))

    return df

Se observa en varias filas la exitencia de tildes.

Decision: Se crea formula para normalizar textos

In [37]:
def normalizar_texto(df):
    return (df.astype("string").str.strip()
            .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("utf-8"))

In [38]:
# Aplicación de formula de limpieza y normalización de texto a variable Clientes
clientes = limpieza_clientes(clientes)
clientes["provincia"] = normalizar_texto(clientes["provincia"])
clientes["situacion_laboral"] = normalizar_texto(clientes["situacion_laboral"])
clientes["estado_civil"] = normalizar_texto(clientes["estado_civil"])

Los datos han sido modificados
   id_cliente fecha_alta  edad               provincia     estado_civil  \
0      C00001 2022-01-02    28               BARCELONA           CASADO   
1      C00002 2022-01-03    45               BARCELONA          SOLTERO   
2      C00003 2022-01-04    60                 SEVILLA  PAREJA DE HECHO   
3      C00004 2022-01-04    26                     BCN           CASADO   
4      C00005 2022-01-05    34                A CORUÑA           CASADO   
5      C00006 2022-01-09    32  SANTA CRUZ DE TENERIFE           CASADO   
6      C00007 2022-09-01    32                  MADRID           CASADO   
7      C00008 2022-01-10    48                 SEVILLA            VIUDO   
8      C00009 2022-01-12    59                 SEVILLA          SOLTERO   
9      C00010 2022-01-14    59                  MADRID           CASADO   
10     C00011 2022-01-15    24              PONTEVEDRA           CASADO   
11     C00012 2022-01-17    40                  MADRID          SOLTE

In [39]:
# Busqueda de anomalias en documento Prestamos
print(prestamos.head(15))

   id_prestamo id_cliente fecha_concesion  importe  plazo_meses  tipo_interes  \
0     PR000001     C00002      2022-01-17  19700.0           36          9.38   
1     PR000002     C00002      2022-02-14   4150.0            6          7.53   
2     PR000003     C00019      22/02/2022   3250.0            6          5.25   
3     PR000004     C00020      2022-02-25  16550.0           36          7.82   
4     PR000005     C00018      2022-03-01   3850.0            6          8.13   
5     PR000006     C00042      2022-03-02  18100.0           24          9.76   
6     PR000007     C00046      2022-03-03   6400.0           12         11.56   
7     PR000008     C00044      2022-03-03   2900.0            6          7.17   
8     PR000009     C00053      11/03/2022  51800.0           72         11.98   
9     PR000010     C00056      2022-03-15  13200.0           36         12.12   
10    PR000011     C00045      2022-03-17   8400.0           12         16.63   
11    PR000012     C00042   

In [40]:
def limpieza_prestamos(df):

    df["fecha_concesion"] = pd.to_datetime(df["fecha_concesion"], format="mixed", dayfirst=False)
    df["finalidad"] = df["finalidad"].str.upper().str.strip()
    df["canal_captacion"] = df["canal_captacion"].str.upper().str.strip()
    df["estado"] = df["estado"].str.upper().str.strip()

    print("Se han realizado los cambios")
    print(df.head(15))

    return df

In [41]:
# Aplicamos formula de limpieza a variable Prestamos
prestamos = limpieza_prestamos(prestamos)

Se han realizado los cambios
   id_prestamo id_cliente fecha_concesion  importe  plazo_meses  tipo_interes  \
0     PR000001     C00002      2022-01-17  19700.0           36          9.38   
1     PR000002     C00002      2022-02-14   4150.0            6          7.53   
2     PR000003     C00019      2022-02-22   3250.0            6          5.25   
3     PR000004     C00020      2022-02-25  16550.0           36          7.82   
4     PR000005     C00018      2022-03-01   3850.0            6          8.13   
5     PR000006     C00042      2022-03-02  18100.0           24          9.76   
6     PR000007     C00046      2022-03-03   6400.0           12         11.56   
7     PR000008     C00044      2022-03-03   2900.0            6          7.17   
8     PR000009     C00053      2022-11-03  51800.0           72         11.98   
9     PR000010     C00056      2022-03-15  13200.0           36         12.12   
10    PR000011     C00045      2022-03-17   8400.0           12         16.63   

In [42]:
# Busqueda anomalias en documento Pagos
print(pagos.head(15))

      id_pago id_prestamo fecha_vencimiento  fecha_pago  importe_cuota  \
0   PG0000001    PR000001        2022-02-17  2022-02-17         629.94   
1   PG0000002    PR000001        2022-03-17  2022-03-18         629.94   
2   PG0000003    PR000001        2022-04-17  2022-04-17         629.94   
3   PG0000004    PR000001        2022-05-17  2022-05-17         629.94   
4   PG0000005    PR000001        2022-06-17  2022-06-17         629.94   
5   PG0000006    PR000001        2022-07-17  2022-07-17         629.94   
6   PG0000007    PR000001        2022-08-17  2022-08-18         629.94   
7   PG0000008    PR000001        2022-09-17  2022-09-18         629.94   
8   PG0000009    PR000001        2022-10-17  20-10-2022         629.94   
9   PG0000010    PR000001        2022-11-17  2022-11-17         629.94   
10  PG0000011    PR000001        2022-12-17  2022-12-17         629.94   
11  PG0000012    PR000001        2023-01-17  2023-01-17         629.94   
12  PG0000013    PR000001        2023-

Se observan anomalias de formato en columna fecha_pago

Decisión: no hace falta crear formula

In [43]:
# Script conversion fecha formato EEUU
pagos["fecha_pago"] = pd.to_datetime(pagos["fecha_pago"], format="mixed", dayfirst=False)

Una vez realizada la limpieza, procedemos a eliminar los datos huerfanos que hayan quedado en Pagos y Prestamos debido a la eliminación de anomalias en Clientes

In [44]:
# 1. Eliminación huerfanos Prestamos
prestamos = prestamos[prestamos["id_cliente"].isin(clientes["id_cliente"])]

# 2. Elimiación huerfanos Pagos
pagos = pagos[pagos["id_prestamo"].isin(prestamos["id_prestamo"])]

print("Huerfanos de documentos Prestamos y Pagos eliminados")

Huerfanos de documentos Prestamos y Pagos eliminados


In [55]:
# Limpieza tipo de interes < 1
mal = prestamos["tipo_interes"] < 1

# Observamos cuandos datos erroneos existen
print(mal.sum())

# Modificamos los valores erróneos
prestamos.loc[mal, "tipo_interes"] *= 100

0
